# Explore a pixel through time
Choose a latitude/longitude within the footprint. The nearest native pixel is
located using geodesic distance. We verify that geolocation stays constant
across scenes before reusing its row and column.
The example location is simply a sample pixel; no incident identification is implied.


In [ ]:
from pathlib import Path
import os, sys
REPO = Path.cwd() if (Path.cwd() / 'examples').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / 'examples'))
from flare_sample import data_root, scenes, open_scene, false_rgb, nearest_pixel, interactive_map
DATA = data_root(os.environ.get('FLARE_DATA_DIR', str(REPO / 'data')))
FORMAT = 'netcdf'  # Public sample is included; use 'zarr' for an approved full download.
TAGS = scenes(DATA, FORMAT)
OUTPUT = REPO / 'outputs'
OUTPUT.mkdir(exist_ok=True)
print(f'{len(TAGS)} scenes; {TAGS[0]} through {TAGS[-1]}. All times UTC.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
LATITUDE, LONGITUDE = 44.803333, -120.199167
def series_at(latitude,longitude):
    with open_scene(DATA,TAGS[0],FORMAT) as first:
        row,col=nearest_pixel(first,latitude,longitude)
        lat0,lon0=float(first.lat[row,col]),float(first.lon[row,col])
    records=[]
    for tag in TAGS:
        with open_scene(DATA,tag,FORMAT) as ds:
            assert float(ds.lat[row,col])==lat0 and float(ds.lon[row,col])==lon0
            record={'time':pd.Timestamp(ds.time.values)}
            record.update({k:float(ds[k][row,col]) for k in ['FireScore','BT04','BT05','ObservationConfidence','AtmosphereState']})
            records.append(record)
    frame=pd.DataFrame(records).set_index('time')
    frame.index=frame.index.tz_localize('UTC')
    # Insert NaN rows across the omitted interval so lines do not bridge it.
    plotted=frame.reindex(pd.date_range(frame.index.min(),frame.index.max(),freq='5min'))
    fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True,constrained_layout=True)
    plotted[['BT04','BT05']].plot(ax=axes[0],marker='o',ylabel='Brightness temperature (K)')
    plotted.FireScore.plot(ax=axes[1],marker='o',color='darkorange',ylabel='FireScore',ylim=(0,100))
    plotted.ObservationConfidence.plot(ax=axes[2],marker='o',drawstyle='steps-post',ylabel='Observation quality',ylim=(-.2,4.2))
    axes[2].set_xlabel('July 16, 2026 · UTC')
    fig.suptitle(f'Nearest pixel: {lat0:.5f}, {lon0:.5f} · row {row}, col {col}')
    fig.savefig(OUTPUT/'pixel_time_series.png',dpi=160)
    plt.show()
    return frame
series=series_at(LATITUDE,LONGITUDE)
display(series)


## Explore another location
Set coordinates within the sample footprint. The printed pixel location is
the actual sampled center. Observation quality is not a fire probability.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
lat_input=widgets.BoundedFloatText(value=LATITUDE,min=43.453,max=45.734,description='Latitude:')
lon_input=widgets.BoundedFloatText(value=LONGITUDE,min=-124.198,max=-118.469,description='Longitude:')
button=widgets.Button(description='Update time series')
output=widgets.Output()
def update(_):
    with output:
        output.clear_output(wait=True)
        display(series_at(lat_input.value,lon_input.value))
button.on_click(update)
display(widgets.VBox([lat_input,lon_input,button,output]))
